In [288]:
# import numpy as np

# def mn_rotation_to_quaternion(alpha_deg, beta_deg, gamma_deg):
#     """
#     Convert rotations in your m-n coordinate system to a quaternion
    
#     alpha_deg: angle of n-axis relative to x-axis
#     beta_deg: rotation about m-axis
#     gamma_deg: rotation about n-axis
#     """
    
#     # Convert to radians
#     alpha = np.radians(alpha_deg)
#     beta = np.radians(beta_deg)
#     gamma = np.radians(gamma_deg)
    
#     # Define your m and n axes in terms of x,y,z coordinates
#     # n-axis direction vector
#     n_axis = np.array([np.cos(alpha), np.sin(alpha), 0])
    
#     # m-axis direction vector (perpendicular to n, in xy plane)
#     m_axis = np.array([-np.sin(alpha), np.cos(alpha), 0])
    
#     # Create rotation quaternions for each axis
#     def axis_angle_to_quat(axis, angle):
#         """Convert axis-angle to quaternion"""
#         axis = axis / np.linalg.norm(axis)  # normalize
#         half_angle = angle / 2
#         w = np.cos(half_angle)
#         xyz = axis * np.sin(half_angle)
#         return np.array([w, xyz[0], xyz[1], xyz[2]])
    
#     # Quaternions for each rotation
#     q_m = axis_angle_to_quat(m_axis, beta)  # rotation about m-axis
#     q_n = axis_angle_to_quat(n_axis, gamma) # rotation about n-axis
    
#     # Multiply quaternions (order: first m, then n)
#     def quat_multiply(q1, q2):
#         w1, x1, y1, z1 = q1
#         w2, x2, y2, z2 = q2
#         return np.array([
#             w1*w2 - x1*x2 - y1*y2 - z1*z2,
#             w1*x2 + x1*w2 + y1*z2 - z1*y2,
#             w1*y2 - x1*z2 + y1*w2 + z1*x2,
#             w1*z2 + x1*y2 - y1*x2 + z1*w2
#         ])
    
#     # Combined rotation
#     final_quat = quat_multiply(q_m, q_n)
#     return final_quat

In [289]:
import numpy as np

def mn_rotation_to_quaternion(alpha_deg, beta_deg, gamma_deg, omega_m=0, omega_n=0):
    """
    Convert rotations and angular velocities in your m-n coordinate system
    
    alpha_deg: angle of n-axis relative to x-axis
    beta_deg: rotation about m-axis
    gamma_deg: rotation about n-axis
    omega_m: angular velocity about m-axis (rad/s) - optional
    omega_n: angular velocity about n-axis (rad/s) - optional
    
    Returns: (quaternion, angular_velocity_world)
    """
    
    # Convert to radians
    alpha = np.radians(alpha_deg)
    beta = np.radians(beta_deg)
    gamma = np.radians(gamma_deg)
    
    # Define your m and n axes in terms of x,y,z coordinates
    # n-axis direction vector
    n_axis = np.array([np.cos(alpha), np.sin(alpha), 0])
    
    # m-axis direction vector (perpendicular to n, in xy plane)
    m_axis = np.array([-np.sin(alpha), np.cos(alpha), 0])
    
    # Create rotation quaternions for each axis
    def axis_angle_to_quat(axis, angle):
        """Convert axis-angle to quaternion"""
        axis = axis / np.linalg.norm(axis)  # normalize
        half_angle = angle / 2
        w = np.cos(half_angle)
        xyz = axis * np.sin(half_angle)
        return np.array([w, xyz[0], xyz[1], xyz[2]])
    
    # Quaternions for each rotation
    q_m = axis_angle_to_quat(m_axis, beta)  # rotation about m-axis
    q_n = axis_angle_to_quat(n_axis, gamma) # rotation about n-axis
    
    # Multiply quaternions (order: first m, then n)
    def quat_multiply(q1, q2):
        w1, x1, y1, z1 = q1
        w2, x2, y2, z2 = q2
        return np.array([
            w1*w2 - x1*x2 - y1*y2 - z1*z2,
            w1*x2 + x1*w2 + y1*z2 - z1*y2,
            w1*y2 - x1*z2 + y1*w2 + z1*x2,
            w1*z2 + x1*y2 - y1*x2 + z1*w2
        ])
    
    # Combined rotation
    final_quat = quat_multiply(q_m, q_n)
    
    # Calculate angular velocity (even if zero)
    omega_world = omega_m * m_axis + omega_n * n_axis
    
    return final_quat, omega_world

In [290]:
import mujoco
import mujoco.viewer
import time

#def jump(hv, vv, d)

# Load the world model
model = mujoco.MjModel.from_xml_path("world1.xml")
data = mujoco.MjData(model)

hv = 8
vv = -5
d = 0
a = 45
b = 10
avm = -4
g = -30
avn = 6

data.qpos[0:3] = [-2.0, 1.5, 1]
quaternion, omega = mn_rotation_to_quaternion(a, b, g, avm, avn)
data.qpos[3:7] = quaternion
data.qvel[0:3] = [hv*np.cos(np.deg2rad(a)), -hv*np.sin(np.deg2rad(a)), vv]
data.qvel[3:6] = omega

# Create viewer with keyboard callback
paused = True

def key_callback(keycode):
    global paused
    if keycode == 32:  # Spacebar key code
        paused = not paused
        if paused:
            print("Simulation PAUSED - Press SPACEBAR to resume")
        else:
            print("Simulation STARTED - Press SPACEBAR to pause")

# Launch the viewer (non-passive for keyboard support)
with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    # Set camera to look down and forward at the bar
    viewer.cam.lookat[0] = 0    # Look at X=0 (center of bar)
    viewer.cam.lookat[1] = 0    # Look at Y=0 (center of bar)  
    viewer.cam.lookat[2] = 1.5  # Look at Z=1.5 (slightly below bar)
    
    viewer.cam.distance = 12     # Distance from the look-at point
    viewer.cam.elevation = -20  # Look down at -20 degrees
    viewer.cam.azimuth = 0      # Face forward (0 degrees)
    
    print("Rod is ready! Press SPACEBAR in the viewer window to start/pause simulation")
    
    while viewer.is_running():
        if not paused:
            mujoco.mj_step(model, data)
        viewer.sync()
        time.sleep(0.01)  # Prevent excessive CPU usage when paused

Rod is ready! Press SPACEBAR in the viewer window to start/pause simulation
Simulation STARTED - Press SPACEBAR to pause
